# MacDougald Lab — Mouse Colony Dashboard
## Reproducible build pipeline

This notebook is the **source of truth** for the live colony dashboard hosted on GitHub Pages.
Re-running it end-to-end regenerates `data/colony.json` and `index.html` from the latest exports
of the colony software.

**Inputs (raw, kept local — gitignored):**
- `CageListExcel.xlsx`
- `MacLab - Brian_Mice.xlsx`
- `StrainList.xlsx`
- `Breedings.xlsx`

**User-authored research knowledge (preserved across runs):**
- `data/cohort_plans.json` — cohort goals, target genotypes, expected Mendelian ratios

**Outputs (committed to repo, deployed by GitHub Pages):**
- `data/colony.json` — aggregated dashboard data
- `index.html` — the dashboard itself (data embedded for offline use)

---

## Update workflow

1. Drop the latest exports into the project folder.
2. **Run all cells** in this notebook (or run the two scripts at the bottom from a terminal).
3. Commit `data/colony.json` and `index.html`, push to GitHub.
4. GitHub Pages rebuilds in ~30 seconds; the shared URL never changes.

## 1 · Sanity-check inputs

In [ ]:
from pathlib import Path
import pandas as pd

DIR = Path('.').resolve()
print(f'Working dir: {DIR}')

required = [
    'CageListExcel.xlsx',
    'MacLab - Brian_Mice.xlsx',
    'StrainList.xlsx',
    'Breedings.xlsx',
    'data/cohort_plans.json',
]
for r in required:
    p = DIR / r
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {r}')

## 2 · Build aggregated data → `data/colony.json`

Calls `build_dashboard_data.py`, which:
- Reads all four Excel exports + `data/cohort_plans.json`
- Computes mouse-level enrichment (age buckets, cull flags)
- Aggregates by strain, sex, age, use status
- Builds per-cage summaries (with rack positions parsed)
- Computes cull-candidate pools by criteria
- Writes a single `data/colony.json` for the dashboard to consume

In [ ]:
import subprocess
result = subprocess.run(['python', 'build_dashboard_data.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('build_dashboard_data.py failed')

## 3 · Render the dashboard → `index.html`

Calls `build_dashboard_html.py`, which embeds `data/colony.json` into a single self-contained
HTML file (Apache ECharts via CDN, no other runtime dependencies).

In [ ]:
result = subprocess.run(['python', 'build_dashboard_html.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('build_dashboard_html.py failed')

## 4 · Quick sanity look at the data

In [ ]:
import json
with open('data/colony.json') as f:
    d = json.load(f)

print('=== Summary ===')
for k, v in d['summary'].items():
    print(f'  {k:30s} {v}')

print()
print('=== Strain sizes ===')
for s in d['strain_order']:
    print(f'  {s:14s} {d["strain_counts"][s]:>4d}')

print()
print('=== Cull candidate pools ===')
for k, v in d['cull_summary'].items():
    print(f'  {k:42s} n = {v["count"]}')

## 5 · Preview the cull pool (top 20)

In [ ]:
inv = pd.DataFrame(d['inventory'])
print('Aged > 12 mo, available, non-breeder:')
mask = (inv['age_months'] > 12) & (inv['use'] == 'Available')
display(inv[mask].sort_values('age_months', ascending=False).head(20)[
    ['mouse_id', 'strain', 'sex', 'age_months', 'genotype', 'cage_id']
])

## 6 · How to publish

After running the cells above, the dashboard files are regenerated on disk.
From a terminal in this folder:

```bash
git add data/colony.json index.html
git commit -m "Refresh colony data <YYYY-MM-DD>"
git push
```

The GitHub Pages URL re-deploys automatically. Don't share a new link — the URL is stable.

To update **cohort plans** (target genotypes, ratios, scientific purpose), edit
`data/cohort_plans.json` directly, then rerun this notebook.